In [26]:
import numpy as np
import pandas as pd

In [91]:
online = pd.read_csv("thetaPSD_online_2back_-1580742246_20260602_170802.csv")
online

,Time,Ch01,Ch02,Ch03,Ch04,Ch05
0,0.000,-33147.292969,-0.053302,0.002841,0.000057,1.297218e+03
1,0.004,-32633.068359,-0.463833,0.215141,0.004360,7.610196e+04
2,0.008,-32385.759766,-2.040662,4.164302,0.087646,1.594247e+05
3,0.012,-32712.271484,-6.192164,38.342896,0.854504,1.541661e+06
4,0.016,-33204.988281,-14.807284,219.255692,5.239618,2.142307e+06
...,...,...,...,...,...,...
6289,25.156,-32391.621094,-0.988708,0.977543,1.178679,1.178679e+00
6290,25.160,-32859.156250,-0.898495,0.807294,1.145904,1.178679e+00
6291,25.164,-33150.242188,-0.790242,0.624482,1.119262,1.178679e+00
6292,25.168,-32654.453125,-0.666595,0.444349,1.098625,1.178679e+00


In [92]:
theta_z_csv = pd.read_csv("theta_z_values_test_-1167338989.csv", header= None, names = ["Time", "online z_score"])
theta_z_csv

,Time,online z_score
0,22.996475,0.470496
1,23.032242,0.333673
2,23.074748,0.923454
3,23.109516,1.081116
4,23.152833,1.798322
...,...,...
604,52.372162,5.840897
605,52.418367,4.188085
606,52.451562,4.443793
607,52.495136,5.658613


In [93]:
merged_df = pd.merge_asof(
    left=online.sort_values("Time"),
    right=theta_z_csv.sort_values("Time"),
    on="Time",
    direction="nearest",
    tolerance=0.01   # 50 ms
)
merged_df

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,online z_score
0,0.000,-33147.292969,-0.053302,0.002841,0.000057,1.297218e+03,NaN
1,0.004,-32633.068359,-0.463833,0.215141,0.004360,7.610196e+04,NaN
2,0.008,-32385.759766,-2.040662,4.164302,0.087646,1.594247e+05,NaN
3,0.012,-32712.271484,-6.192164,38.342896,0.854504,1.541661e+06,NaN
4,0.016,-33204.988281,-14.807284,219.255692,5.239618,2.142307e+06,NaN
...,...,...,...,...,...,...,...
6289,25.156,-32391.621094,-0.988708,0.977543,1.178679,1.178679e+00,1.061996
6290,25.160,-32859.156250,-0.898495,0.807294,1.145904,1.178679e+00,1.061996
6291,25.164,-33150.242188,-0.790242,0.624482,1.119262,1.178679e+00,1.061996
6292,25.168,-32654.453125,-0.666595,0.444349,1.098625,1.178679e+00,NaN


In [94]:
buffer = []
offline_z = []
last_theta_val = None

MU = 2.32
SIGMA = 4.18
MAD_THRESHOLD = 6
THETA_THRESHOLD_Z = 1.5
INITIAL_CUTOFF = 100.0
BUFFER_SIZE = 500
COOLDOWN_TIME = 5

# Ch05 = index 4
channel = " Ch05"

for i in range(len(merged_df)):

    theta_val = merged_df[channel].iloc[i]   # MUST match online
    if last_theta_val is not None and theta_val == last_theta_val:
        offline_z.append(np.nan) 
        continue
    last_theta_val = theta_val
    # Warm-up
    if len(buffer) <= 450:
        if theta_val < INITIAL_CUTOFF:
            buffer.append(theta_val)
        offline_z.append(np.nan)
        continue

    # Rolling buffer
    if len(buffer) > BUFFER_SIZE:
        buffer.pop(0)

    arr = np.array(buffer)
    median = np.median(arr)
    mad = np.median(np.abs(arr - median)) + 1e-6

    # Artifact rejection
    z_art = abs(theta_val - median) / mad
    if z_art > MAD_THRESHOLD:
        offline_z.append(np.nan)
        continue

    # Clean sample
    buffer.append(theta_val)

    # Z-score
    theta_z = abs(theta_val - MU) / SIGMA
    offline_z.append(theta_z)

merged_df["offline z-score"] = offline_z


In [95]:
LIFU = np.zeros(len(merged_df))
last_trigger_time = -999

for i in range(len(merged_df)):
    theta_z = merged_df["offline z-score"][i]
    now = merged_df["Time"][i]

    if theta_z > THETA_THRESHOLD_Z and theta_z < MAD_THRESHOLD and (now - last_trigger_time > COOLDOWN_TIME):
        LIFU[i] = 1.0
        last_trigger_time = now


In [96]:
merged_df["LIFU"] = LIFU
merged_df

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,online z_score,offline z-score,LIFU
0,0.000,-33147.292969,-0.053302,0.002841,0.000057,1.297218e+03,NaN,NaN,0.0
1,0.004,-32633.068359,-0.463833,0.215141,0.004360,7.610196e+04,NaN,NaN,0.0
2,0.008,-32385.759766,-2.040662,4.164302,0.087646,1.594247e+05,NaN,NaN,0.0
3,0.012,-32712.271484,-6.192164,38.342896,0.854504,1.541661e+06,NaN,NaN,0.0
4,0.016,-33204.988281,-14.807284,219.255692,5.239618,2.142307e+06,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...
6289,25.156,-32391.621094,-0.988708,0.977543,1.178679,1.178679e+00,1.061996,0.273043,0.0
6290,25.160,-32859.156250,-0.898495,0.807294,1.145904,1.178679e+00,1.061996,NaN,0.0
6291,25.164,-33150.242188,-0.790242,0.624482,1.119262,1.178679e+00,1.061996,NaN,0.0
6292,25.168,-32654.453125,-0.666595,0.444349,1.098625,1.178679e+00,NaN,NaN,0.0


In [97]:
mask_lifu = merged_df["LIFU"] == 1.0
lifu_on = merged_df[mask_lifu]
lifu_on

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,online z_score,offline z-score,LIFU
5669,22.676,-32977.800781,3.669715,13.46681,10.526736,10.526736,NaN,1.963334,1.0
